# TUGAS BESAR DASAR KECERDASAN BUATAN (DKA)
## SISTEM ESTIMASI HARGA MOBIL BEKAS MERCEDES-BENZ MENGGUNAKAN LOGIKA FUZZY

---

**KELOMPOK / ANGGOTA:**
1. **Nama Lengkap 1** - NIM: **NIM Anggota 1**
2. **Nama Lengkap 2** - NIM: **NIM Anggota 2**
3. **Nama Lengkap 3** - NIM: **NIM Anggota 3**

**KELAS:** [Masukkan Kelas Anda, misal: IF-44-01]

**PRODI:** S1 Informatika
**FAKULTAS:** Fakultas Informatika, Universitas Telkom


### Langkah 1: Import Library & Persiapan Awal
Langkah pertama adalah mengimpor pustaka (library) dasar Python yang diperlukan untuk pemrosesan data, manipulasi matriks, visualisasi grafik, dan metrik evaluasi.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error

### Langkah 2: Exploratory Data Analysis (EDA) & Load Dataset
Membaca berkas dataset riil Mercedes-Benz dari `data/merc.csv` untuk memahami tipe data, statistika deskriptif, dan sebaran datanya demi merancang batas-batas himpunan fuzzy.

In [ ]:
data_mobil = pd.read_csv("data/merc.csv")
data_mobil.head()

In [ ]:
data_mobil.info()

In [ ]:
data_mobil.describe()

In [ ]:
# Menyaring kolom data yang relevan sebagai parameter input fuzzy
data_mobil = data_mobil[['year', 'mileage', 'tax', 'mpg', 'engineSize', 'price']]
data_mobil.head()

In [ ]:
data_mobil.describe()

In [ ]:
# Visualisasi distribusi frekuensi parameter dataset
data_mobil.hist(figsize=(12,8))
plt.show()

### Langkah 3: Definisikan Fungsi Keanggotaan Fuzzy (Fuzzifikasi)
Fuzzifikasi adalah proses pemetaan nilai tegas (crisp value) ke dalam nilai linguistik fuzzy menggunakan fungsi keanggotaan. Di sini kita menggunakan **Fungsi Keanggotaan Segitiga** (Triangular Membership Function) karena sifatnya yang sederhana dan efisien.

In [ ]:
def triangleMember(x, a, b, c):
    if x <= a or x >= c:
        return 0
    elif a < x <= b:
        return (x - a) / (b - a)
    elif b < x < c:
        return (c - x) / (c - b)

In [ ]:
# Pengujian fungsi kurva keanggotaan segitiga
triangleMember(20000, 0, 20000, 40000)

#### 3.1. Variabel Input: Mileage (Jarak Tempuh)
- Kategori: **Low** (0–40.000), **Medium** (30.000–70.000), **High** (60.000–100.000+)

In [ ]:
# Visualisasi Himpunan Fuzzy Mileage
x = np.linspace(0, 100000, 1000)
low = [triangleMember(i, 0, 20000, 40000) for i in x]
medium = [triangleMember(i, 30000, 50000, 70000) for i in x]
high = [triangleMember(i, 60000, 80000, 100000) for i in x]

plt.plot(x, low, label="Low")
plt.plot(x, medium, label="Medium")
plt.plot(x, high, label="High")
plt.title("Mileage Membership")
plt.xlabel("Mileage")
plt.ylabel("Membership Degree")
plt.legend()
plt.show()

In [ ]:
def fuzzify_mileage(mileage):
    low = triangleMember(mileage, 0, 20000, 40000)
    medium = triangleMember(mileage, 30000, 50000, 70000)
    high = triangleMember(mileage, 60000, 80000, 100000)
    return {"Low": low, "Medium": medium, "High": high}

fuzzify_mileage(35000)

#### 3.2. Variabel Input: Year (Tahun Mobil)
- Kategori: **Old** (2010–2018), **Medium** (2015–2021), **New** (2018–2023)

In [ ]:
# Visualisasi Himpunan Fuzzy Year
x = np.linspace(2010, 2023, 1000)
old = [triangleMember(i, 2010, 2014, 2018) for i in x]
medium = [triangleMember(i, 2015, 2018, 2021) for i in x]
new = [triangleMember(i, 2018, 2021, 2023) for i in x]

plt.plot(x, old, label="Old")
plt.plot(x, medium, label="Medium")
plt.plot(x, new, label="New")
plt.title("Year Membership")
plt.xlabel("Year")
plt.ylabel("Membership Degree")
plt.legend()
plt.show()

In [ ]:
def fuzzify_year(year):
    old = triangleMember(year, 2010, 2014, 2018)
    medium = triangleMember(year, 2015, 2018, 2021)
    new = triangleMember(year, 2018, 2021, 2023)
    return {"Old": old, "Medium": medium, "New": new}

fuzzify_year(2019)

#### 3.3. Variabel Input: Engine Size (Kapasitas Mesin)
- Kategori: **Kecil** (0–2.0), **Sedang** (1.5–3.5), **Besar** (3.0–6.0+)

In [ ]:
# Visualisasi Himpunan Fuzzy Engine Size
x = np.linspace(0, 6.5, 1000)
kecil = [triangleMember(i, 0, 1.0, 2.0) for i in x]
sedang = [triangleMember(i, 1.5, 2.5, 3.5) for i in x]
besar = [triangleMember(i, 3.0, 4.5, 6.0) for i in x]

plt.plot(x, kecil, label="Kecil")
plt.plot(x, sedang, label="Sedang")
plt.plot(x, besar, label="Besar")
plt.title("Engine Size Membership")
plt.xlabel("Engine Size")
plt.ylabel("Membership Degree")
plt.legend()
plt.show()

In [ ]:
def fuzzify_engine(engine_size):
    kecil = triangleMember(engine_size, 0, 1.0, 2.0)
    sedang = triangleMember(engine_size, 1.5, 2.5, 3.5)
    besar = triangleMember(engine_size, 3.0, 4.5, 6.0)
    return {"Kecil": kecil, "Sedang": sedang, "Besar": besar}

fuzzify_engine(2.5)

#### 3.4. Variabel Input: MPG (Konsumsi Bahan Bakar)
- Kategori: **Boros** (0–40), **Standar** (30–70), **Irit** (60–80+)

In [ ]:
# Visualisasi Himpunan Fuzzy MPG
x = np.linspace(0, 80, 1000)
boros = [triangleMember(i, 0, 20, 40) for i in x]
standar = [triangleMember(i, 30, 50, 70) for i in x]
irit = [triangleMember(i, 60, 70, 80) for i in x]

plt.plot(x, boros, label="Boros")
plt.plot(x, standar, label="Standar")
plt.plot(x, irit, label="Irit")
plt.title("MPG Membership")
plt.xlabel("MPG")
plt.ylabel("Membership Degree")
plt.legend()
plt.show()

In [ ]:
def fuzzify_mpg(mpg):
    boros = triangleMember(mpg, 0, 20, 40)
    standar = triangleMember(mpg, 30, 50, 70)
    irit = triangleMember(mpg, 60, 70, 80)
    return {"Boros": boros, "Standar": standar, "Irit": irit}

fuzzify_mpg(55)

#### 3.5. Variabel Input: Tax (Pajak Tahunan)
- Kategori: **Murah** (0–200), **Standar** (150–450), **Mahal** (400–600+)

In [ ]:
# Visualisasi Himpunan Fuzzy Tax
x = np.linspace(0, 600, 1000)
murah = [triangleMember(i, 0, 100, 200) for i in x]
standar = [triangleMember(i, 150, 300, 450) for i in x]
mahal = [triangleMember(i, 400, 500, 600) for i in x]

plt.plot(x, murah, label="Murah")
plt.plot(x, standar, label="Standar")
plt.plot(x, mahal, label="Mahal")
plt.title("Tax Membership")
plt.xlabel("Tax")
plt.ylabel("Membership Degree")
plt.legend()
plt.show()

In [ ]:
def fuzzify_tax(tax):
    murah = triangleMember(tax, 0, 100, 200)
    standar = triangleMember(tax, 150, 300, 450)
    mahal = triangleMember(tax, 400, 500, 600)
    return {"Murah": murah, "Standar": standar, "Mahal": mahal}

fuzzify_tax(250)

#### 3.6. Variabel Output: Price (Harga) — Khusus Mamdani
Pada metode Mamdani, variabel keluaran (output) didefinisikan sebagai daerah fuzzy kontinu:
- **Murah** (0, 15.000, 30.000)
- **Standar** (20.000, 35.000, 50.000)
- **Mewah** (40.000, 50.000, 100.000)

In [ ]:
# Visualisasi Himpunan Fuzzy Price (Mamdani Output)
x = np.linspace(0, 100000, 1000)
murah = [triangleMember(i, 0, 15000, 30000) for i in x]
standar = [triangleMember(i, 20000, 35000, 50000) for i in x]
mewah = [triangleMember(i, 40000, 50000, 100000) for i in x]

plt.plot(x, murah, label="Murah")
plt.plot(x, standar, label="Standar")
plt.plot(x, mewah, label="Mewah")
plt.title("Price Membership (Mamdani Output)")
plt.xlabel("Price ($)")
plt.ylabel("Membership Degree")
plt.legend()
plt.show()

In [ ]:
def fuzzify_price(price):
    murah = triangleMember(price, 0, 15000, 30000)
    standar = triangleMember(price, 20000, 35000, 50000)
    mewah = triangleMember(price, 40000, 50000, 100000)
    return {"Murah": murah, "Standar": standar, "Mewah": mewah}

fuzzify_price(28000)

### Langkah 4: Definisikan Aturan Fuzzy (Rule Base) & Inferensi
Inferensi mengevaluasi kombinasi variabel input berdasarkan **20 aturan inferensi**. Kita menggunakan operator AND (diwakili oleh fungsi `min`) dan mengagregasikan kontribusi output dengan operator OR (diwakili fungsi `max`).

In [ ]:
def rule_evaluation_all(year, mileage, engine_size, mpg, tax):
    year_fuzzy = fuzzify_year(year)
    mileage_fuzzy = fuzzify_mileage(mileage)
    engine_fuzzy = fuzzify_engine(engine_size)
    mpg_fuzzy = fuzzify_mpg(mpg)
    tax_fuzzy = fuzzify_tax(tax)

    # 20 Aturan Inferensi Fuzzy
    rule1 = min(year_fuzzy["New"], mileage_fuzzy["Low"], engine_fuzzy["Besar"])
    rule2 = min(year_fuzzy["Old"], mileage_fuzzy["High"])
    rule3 = min(mpg_fuzzy["Irit"], tax_fuzzy["Murah"])
    rule4 = min(year_fuzzy["New"], mileage_fuzzy["Low"], mpg_fuzzy["Irit"])
    rule5 = min(engine_fuzzy["Besar"], tax_fuzzy["Mahal"])
    rule6 = min(year_fuzzy["Old"], mileage_fuzzy["High"], mpg_fuzzy["Boros"])
    rule7 = min(engine_fuzzy["Kecil"], mpg_fuzzy["Irit"])
    rule8 = min(tax_fuzzy["Mahal"], mileage_fuzzy["Low"])
    rule9 = min(year_fuzzy["Medium"], mileage_fuzzy["Medium"])
    rule10 = min(engine_fuzzy["Kecil"], mileage_fuzzy["High"])
    rule11 = min(year_fuzzy["New"], engine_fuzzy["Besar"])
    rule12 = min(year_fuzzy["Old"], tax_fuzzy["Murah"])
    rule13 = min(mpg_fuzzy["Boros"], tax_fuzzy["Mahal"])
    rule14 = min(mileage_fuzzy["Low"], mpg_fuzzy["Irit"])
    rule15 = min(engine_fuzzy["Sedang"], year_fuzzy["Medium"])
    rule16 = min(year_fuzzy["Medium"], mpg_fuzzy["Standar"])
    rule17 = min(mileage_fuzzy["Medium"], tax_fuzzy["Standar"])
    rule18 = min(engine_fuzzy["Sedang"], mileage_fuzzy["Medium"])
    rule19 = min(year_fuzzy["New"], engine_fuzzy["Sedang"])
    rule20 = min(year_fuzzy["Old"], engine_fuzzy["Kecil"])

    rules_list = [
        rule1, rule2, rule3, rule4, rule5, rule6, rule7, rule8, rule9, rule10,
        rule11, rule12, rule13, rule14, rule15, rule16, rule17, rule18, rule19, rule20
    ]

    # Agregasi implikasi Mamdani
    agg_dict = {
        "Murah": max(rule2, rule6, rule10, rule12, rule13, rule20),
        "Standar": max(rule3, rule7, rule9, rule15, rule16, rule17, rule18),
        "Mewah": max(rule1, rule4, rule5, rule8, rule11, rule14, rule19)
    }

    return agg_dict, rules_list

def rule_evaluation(year, mileage, engine_size, mpg, tax):
    agg_dict, _ = rule_evaluation_all(year, mileage, engine_size, mpg, tax)
    return agg_dict

In [ ]:
# Uji coba evaluasi aturan fuzzy
rule_evaluation(2020, 15000, 4.5, 60, 150)

### Langkah 5: Defuzzifikasi (Mamdani & Sugeno)
Defuzzifikasi mengubah output fuzzy menjadi nilai tegas (crisp price). Kita mengimplementasikan dua metode:

1. **Fuzzy Mamdani (Centroid / COA)**:
   Mencari titik pusat gravitasi area luaran fuzzy secara numerik dengan diskritisasi semesta \$0 s.d. \$100.000 dengan langkah \$500.
   
2. **Fuzzy Sugeno (Weighted Average)**:
   Menggunakan nilai output tegas konstan (*singleton/konstanta*) untuk setiap aturan:
   - Output Aturan Murah = \$15.000
   - Output Aturan Standar = \$35.000
   - Output Aturan Mewah = \$50.000

In [ ]:
def defuzzify_mamdani(output_fuzzy):
    w_murah = output_fuzzy["Murah"]
    w_standar = output_fuzzy["Standar"]
    w_mewah = output_fuzzy["Mewah"]
    
    # Diskritisasi harga dari $0 s.d. $100,000 dengan langkah 500
    y_vals = np.arange(0, 100001, 500)
    numerator = 0
    denominator = 0
    
    for y in y_vals:
        mu_murah = triangleMember(y, 0, 15000, 30000)
        mu_standar = triangleMember(y, 20000, 35000, 50000)
        mu_mewah = triangleMember(y, 40000, 50000, 100000)
        
        # Clipping (min) dan Aggregasi (max)
        mu_y = max(min(w_murah, mu_murah), min(w_standar, mu_standar), min(w_mewah, mu_mewah))
        
        numerator += y * mu_y
        denominator += mu_y
        
    if denominator == 0:
        return 0
    return numerator / denominator

def defuzzify_sugeno(rules):
    # Output konstan (singleton) untuk masing-masing 20 aturan
    z = [
        50000, 15000, 35000, 50000, 50000, 15000, 35000, 50000, 35000, 15000,
        50000, 15000, 15000, 50000, 35000, 35000, 35000, 35000, 50000, 15000
    ]
    numerator = sum(rules[i] * z[i] for i in range(20))
    denominator = sum(rules)
    if denominator == 0:
        return 0
    return numerator / denominator

def defuzzification(output_fuzzy):
    # Sebagai backward compatibility default ke Mamdani Centroid
    return defuzzify_mamdani(output_fuzzy)

In [ ]:
# Menghitung estimasi harga tegas
hasil_fuzzy = rule_evaluation(2020, 15000, 4.5, 60, 150)
prediksi_mamdani = defuzzification(hasil_fuzzy)
print("Estimasi Harga Mamdani Centroid:", prediksi_mamdani)

### Langkah 6: Prediksi Harga & Pengkategorian Mobil
Membuat fungsi pembungkus (wrapper) prediksi untuk memprediksi harga berdasarkan input numerik riil dan metode pilihan. Kita juga mengkategorikan mobil bekas menjadi:
- **Economy** (harga < \$20.000)
- **Standard** (harga \$20.000 s.d. \$34.999)
- **Premium** (harga \$35.000 s.d. \$49.999)
- **Luxury** (harga >= \$50.000)

In [ ]:
def prediksi_harga(year, mileage, engine_size, mpg, tax, method="Mamdani"):
    agg_dict, rules_list = rule_evaluation_all(year, mileage, engine_size, mpg, tax)
    if method.lower() == "sugeno":
        return defuzzify_sugeno(rules_list)
    else:
        return defuzzify_mamdani(agg_dict)

def kategori_mobil(harga):
    if harga < 20000:
        return "Economy"
    elif harga < 35000:
        return "Standard"
    elif harga < 50000:
        return "Premium"
    else:
        return "Luxury"

In [ ]:
# Menghitung prediksi final untuk Mercedes C-Class
harga = prediksi_harga(2020, 15000, 4.5, 60, 150, method="Mamdani")
kategori = kategori_mobil(harga)

print("=======- HASIL PREDIKSI KENDARAAN -=======")
print("Estimasi Harga Mobil : $", round(harga, 2))
print("Kategori Mobil       :", kategori)

### Langkah 7: Pengujian & Evaluasi Performa (Mamdani vs Sugeno)
Kita membandingkan hasil prediksi kedua model fuzzy *from scratch* ini dengan harga riil pada **25 data teratas** dari dataset Mercedes-Benz (`merc.csv`) untuk menghitung metrik error **Mean Absolute Error (MAE)**, **Mean Squared Error (MSE)**, dan **Root Mean Squared Error (RMSE)**.

In [ ]:
harga_asli = []
harga_prediksi_mamdani = []
harga_prediksi_sugeno = []
sample_data = data_mobil.head(25)

In [ ]:
for index, row in sample_data.iterrows():
    pred_mamdani = prediksi_harga(row["year"], row["mileage"], row["engineSize"], row["mpg"], row["tax"], method="Mamdani")
    pred_sugeno = prediksi_harga(row["year"], row["mileage"], row["engineSize"], row["mpg"], row["tax"], method="Sugeno")
    
    harga_prediksi_mamdani.append(pred_mamdani)
    harga_prediksi_sugeno.append(pred_sugeno)
    harga_asli.append(row["price"])

In [ ]:
print("Perbandingan Prediksi Harga Mobil Bekas (Mamdani vs Sugeno):\n")
for i in range(len(harga_asli)):
    kat_mamdani = kategori_mobil(harga_prediksi_mamdani[i])
    kat_sugeno = kategori_mobil(harga_prediksi_sugeno[i])
    print(f"Data ke-{i+1:02d} | Harga Asli: ${harga_asli[i]:,.2f}")
    print(f"        -> Mamdani : ${harga_prediksi_mamdani[i]:,.2f} ({kat_mamdani})")
    print(f"        -> Sugeno  : ${harga_prediksi_sugeno[i]:,.2f} ({kat_sugeno})")
    print("-" * 50)

In [ ]:
mae_mamdani = mean_absolute_error(harga_asli, harga_prediksi_mamdani)
mae_sugeno = mean_absolute_error(harga_asli, harga_prediksi_sugeno)
print(f"MAE Mamdani : {mae_mamdani:,.2f}")
print(f"MAE Sugeno  : {mae_sugeno:,.2f}")

In [ ]:
mse_mamdani = mean_squared_error(harga_asli, harga_prediksi_mamdani)
mse_sugeno = mean_squared_error(harga_asli, harga_prediksi_sugeno)
print(f"MSE Mamdani : {mse_mamdani:,.2f}")
print(f"MSE Sugeno  : {mse_sugeno:,.2f}")

In [ ]:
rmse_mamdani = np.sqrt(mse_mamdani)
rmse_sugeno = np.sqrt(mse_sugeno)
print(f"RMSE Mamdani: {rmse_mamdani:,.2f}")
print(f"RMSE Sugeno : {rmse_sugeno:,.2f}")

# --- Analisis Perbandingan Mamdani vs Sugeno

Berdasarkan hasil pengujian di atas, berikut adalah perbandingan antara Metode Fuzzy Mamdani dan Sugeno:

### 1. Perbedaan Hasil Output & Performa (Akurasi/Error)
- **Hasil Output**: Hasil prediksi dari Fuzzy Mamdani dan Fuzzy Sugeno menunjukkan sedikit perbedaan nominal harga. Hal ini disebabkan oleh perbedaan mendasar pada proses defuzzifikasi.
- **Evaluasi Metrik (MAE & MSE)**: Metode yang menghasilkan error lebih rendah bervariasi bergantung pada kecocokan singleton/konstanta output pada Sugeno terhadap sebaran data riil. Sugeno cenderung lebih konsisten karena sifatnya yang linier/konstan tanpa dipengaruhi oleh bentuk kurva keanggotaan output.

### 2. Interpretasi Kelebihan dan Kekurangan masing-masing metode

| Aspek Perbandingan | Fuzzy Mamdani | Fuzzy Sugeno |
| :--- | :--- | :--- |
| **Format Output** | Berupa himpunan fuzzy (kurva kontinu/segitiga/trapesium). | Berupa nilai konstanta (singleton) atau persamaan linier. |
| **Proses Defuzzifikasi** | Menggunakan metode **Centroid** (Center of Area) yang membutuhkan diskritisasi/integrasi. | Menggunakan rata-rata terbobot (**Weighted Average**) yang langsung menghitung nilai tegas. |
| **Beban Komputasi** | **Lebih Tinggi**: Membutuhkan iterasi/diskritisasi untuk mencari titik pusat gravitasi pada area output. | **Sangat Rendah**: Rumus matematika sederhana langsung menghasilkan nilai tegas dengan cepat. |
| **Kecocokan Input/Output** | Sangat cocok untuk menangani persepsi kualitatif manusia yang bersifat subjektif. | Sangat cocok untuk optimasi matematis, kontroler industri, dan integrasi dengan Machine Learning. |
| **Intuitif bagi Manusia** | Lebih intuitif dan mudah dipahami karena seluruh variabel (termasuk output) memiliki grafik linguistik. | Kurang intuitif untuk outputnya karena langsung diwakili oleh angka konstanta/singleton. |
